# 第5章 回归模型 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]
- [[课程/深度学习/PPT/Chapter2/Chapter2.md]]（仅误差与泛化差距口径）
- [[课程/深度学习/演示代码/Chapter5.py]]
- [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md]]
- [[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md]]
- [[课程/深度学习/作业/Homework03/Homework03.md]]
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## Q501 立方体与球体：均匀的是体积，不是半径

**新编，依据** [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#二、训练问题与样本采样]]、[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#第5章 回归模型]] C5-01。

一句话来说：立方体逐坐标均匀即可，球体要让半径满足体积比例。

实现 `sample_cube(n,d,seed)`、`sample_ball(n,d,R,seed)`，均返回 `(n,d)` CPU float32 Tensor。
立方体为 `[-1,1]^d`；球内方向为独立标准正态向量归一化，半径为 `R*U**(1/d)`。
要求 n≥0、d≥1、R>0，非法参数抛 ValueError；n=0返回正确空形状。用同seed复现。
取d=3、n=5000，报告球内半径≤R/2的比例，理论为1/8；不能用U直接当半径。

**自检接口／需保存的结果**

- 函数应支持省略可选参数：`sample_cube(n,d,seed=501)`、`sample_ball(n,d,R=1.0,seed=501)`，以便自检直接调用默认值。
- `ball501`：半径 **R=2**、d=3、n=5000的球内样本，CPU float32 Tensor，形状`(5000,3)`；`fraction501`：这批样本中到原点距离不超过1的比例，Python float。


<details><summary>提示（先尝试再展开）</summary>

独立 Generator 保证函数复现不被其他题消耗的随机数影响。体积比例是(r/R)^d。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def sample_cube(n, d, seed=501):
    if n < 0 or d < 1:
        raise ValueError("invalid sample size/dimension")
    return 2 * torch.rand(n, d, generator=torch.Generator().manual_seed(seed)) - 1

def sample_ball(n, d, R=1.0, seed=501):
    if n < 0 or d < 1 or R <= 0:
        raise ValueError("invalid ball")
    g = torch.Generator().manual_seed(seed)
    z = torch.randn(n, d, generator=g)
    norms = z.norm(dim=1, keepdim=True)
    # A zero normal vector has probability zero mathematically; handle it in finite arithmetic.
    zero = norms.squeeze(-1) == 0
    while zero.any():
        z[zero] = torch.randn(int(zero.sum()), d, generator=g)
        norms = z.norm(dim=1, keepdim=True)
        zero = norms.squeeze(-1) == 0
    r = R * torch.rand(n, 1, generator=g).pow(1 / d)
    return z / norms * r

ball501 = sample_ball(5000, 3, 2.0)
fraction501 = (ball501.norm(dim=1) <= 1).float().mean().item()
print("Inside half-radius:", fraction501, "theory:", 1/8)


### 解析与点评

若半径直接均匀，半径≤R/2的比例约1/2而非1/8。n=0时广播运算仍可返回空矩阵；检查距离不能证明分布均匀，因此另检查一个有诊断力的径向概率。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert sample_cube(0, 4).shape == (0, 4)
assert sample_ball(0, 2).shape == (0, 2)
assert torch.equal(sample_ball(17, 3, seed=12), sample_ball(17, 3, seed=12))
assert torch.all(sample_cube(100, 5).abs() <= 1)
assert torch.all(ball501.norm(dim=1) <= 2.000001)
assert abs(fraction501 - 0.125) < 0.035  # broad distribution check, not exact floats
for call in (lambda: sample_ball(3, 0), lambda: sample_ball(3, 2, 0)):
    try:
        call()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input not rejected")

print("自检通过")


## Q502 Module 与 Sequential：形状、层号和参数量

**新编/演示改编，依据** [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#七、PyTorch 网络实现]]、[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C5-06 批量输出、参数提取与层索引]]。

实现自定义类 `Regressor5(d=5,width=10)`，结构5→10→10→1（两隐藏ReLU，输出不激活），另建同结构Sequential并复制各层权重偏置，验证输出一致。
实现 `count_parameters(net)`；报告(深度,宽度)=(3,10),(3,20),(5,10)参数量181、561、401。
定义通用 `make_reg5(d,width,depth)`。depth计Linear层，不计激活；输出始终 `(N,1)`，N=1时用 `.squeeze(-1)`得到 `(1,)`。
解释Sequential的第1、2、3个Linear索引是0、2、4，十次Linear交替激活时第5、第6个Linear索引是8、10。

**自检接口／需保存的结果**

- `module502`：上述自定义`Regressor5`的d=5、width=10实例；`seq502`：已与它同步参数的同结构`nn.Sequential`，其中第0、2、4项为Linear，第1、3项为ReLU，便于按层检查。
- `x502`：供两网络比较的同一批有限CPU float32输入，形状`(7,5)`；两网络在该输入上的输出均为`(7,1)`。
- `counts502`：按`(depth,width)=(3,10),(3,20),(5,10)`顺序保存参数总数的Python整数列表，长度3。


<details><summary>提示（先尝试再展开）</summary>

Linear(a,b)含ab个权重、b个偏置。用load_state_dict复制每个层，随机种子相同不自动表示两个顺序构造网络相同。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

class Regressor5(nn.Module):
    def __init__(self, d=5, width=10):
        super().__init__()
        self.fc1 = nn.Linear(d, width)
        self.fc2 = nn.Linear(width, width)
        self.fc3 = nn.Linear(width, 1)
    def forward(self, x):
        return self.fc3(torch.relu(self.fc2(torch.relu(self.fc1(x)))))

def make_reg5(d=5, width=10, depth=3):
    if depth < 2:
        raise ValueError("depth >= 2")
    layers, last = [], d
    for _ in range(depth - 1):
        layers.extend([nn.Linear(last, width), nn.ReLU()])
        last = width
    return nn.Sequential(*layers, nn.Linear(width, 1))

def count_parameters(net):
    return sum(p.numel() for p in net.parameters() if p.requires_grad)

module502 = Regressor5()
seq502 = make_reg5()
for old, new in zip((module502.fc1, module502.fc2, module502.fc3),
                    (seq502[0], seq502[2], seq502[4])):
    new.load_state_dict(old.state_dict())
x502 = torch.randn(7, 5)
counts502 = [count_parameters(make_reg5(5, m, l)) for l, m in [(3, 10), (3, 20), (5, 10)]]
print(counts502, "Linear indices:", [0, 2, 4], "later:", [8, 10])


### 解析与点评

每层参数为b(a+1)，所以60+110+11=181。深度增加不一定参数更多，也不保证测试误差更低。裸squeeze可能删除N=1的样本维，明确squeeze(-1)只去输出维。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert torch.allclose(module502(x502), seq502(x502))
assert counts502 == [181, 561, 401]
assert seq502(torch.zeros(1, 5)).shape == (1, 1)
assert seq502(torch.zeros(1, 5)).squeeze(-1).shape == (1,)
assert tuple(seq502[0].weight.shape) == (10, 5)
assert tuple(seq502[4].bias.shape) == (1,)

print("自检通过")


## Q503 MSE、RMSE与课程泛化差距

**新编，依据** [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]“测试”小节、[[课程/深度学习/PPT/Chapter2/Chapter2.md]]“测试集、泛化误差与泛化差距”小节、[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#A-02 MSE与RMSE的口径]]。

实现 `regression_metrics(pred,target)`，同形非空张量→含mse、rmse的Python float字典，拒绝广播。
实现 `regression_report(train_pred,train_y,test_pred,test_y)`，返回train_mse、train_rmse、test_mse、test_rmse、gap_abs、gap_signed。
遵课程口径：**泛化差距gap_abs=abs(test_rmse−train_rmse)**；gap_signed仅作方向诊断，可为负。
用训练残差(1,1)、测试残差(1,3)手核；再用训练残差(3)、测试残差(1)说明负signed差距。
不把MSE与RMSE相减，也不把含正则项的objective当RMSE平方。

**自检接口／需保存的结果**

将两组手核案例的`regression_report`返回值分别保存为`report503`（训练残差为1、1，测试残差为1、3）和`negative503`（训练残差为3，测试残差为1）。二者均为题面约定的六键字典；各值为Python float，不是Tensor或打印字符串。


<details><summary>提示（先尝试再展开）</summary>

先求残差平方的平均，再开平方。两者必须用同一口径；abs与signed应分别命名。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def make_mlp(in_dim, out_dim=1, depth=3, width=100, activation="ReLU"):
    # depth counts Linear layers; hidden layers = depth - 1.
    if depth < 2 or width < 1:
        raise ValueError("depth >= 2 and width >= 1 required")
    activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
    if activation not in activations:
        raise ValueError("unknown activation")
    layers, last = [], in_dim
    for _ in range(depth - 1):
        layers += [nn.Linear(last, width), activations[activation]()]
        last = width
    layers += [nn.Linear(last, out_dim)]
    return nn.Sequential(*layers)

def regression_metrics(pred, target):
    # Deliberately reject broadcasting; squeeze(-1) is done at the caller.
    if pred.shape != target.shape or target.numel() == 0:
        raise ValueError("nonempty, equal shapes required")
    mse = ((pred - target) ** 2).mean().item()
    return {"mse": mse, "rmse": float(np.sqrt(mse))}

def regression_report(train_pred, train_y, test_pred, test_y):
    a = regression_metrics(train_pred, train_y)
    b = regression_metrics(test_pred, test_y)
    signed = b["rmse"] - a["rmse"]
    return {"train_mse": a["mse"], "train_rmse": a["rmse"],
            "test_mse": b["mse"], "test_rmse": b["rmse"],
            "gap_abs": abs(signed), "gap_signed": signed}

report503 = regression_report(torch.tensor([1., 1.]), torch.zeros(2),
                              torch.tensor([1., 3.]), torch.zeros(2))
negative503 = regression_report(torch.tensor([3.]), torch.zeros(1),
                                torch.tensor([1.]), torch.zeros(1))
print(report503)
print("Negative signed difference:", negative503)


### 解析与点评

训练RMSE为1，测试RMSE为sqrt(5)，课程gap为sqrt(5)-1。差距很小并不等于误差小：两边都很差也会gap≈0；有限样本中测试RMSE低于训练也完全可能。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.isclose(report503["train_mse"], 1)
assert np.isclose(report503["test_mse"], 5)
assert np.isclose(report503["gap_abs"], np.sqrt(5)-1)
assert negative503["gap_signed"] == -2 and negative503["gap_abs"] == 2
try:
    regression_metrics(torch.zeros(3, 1), torch.zeros(3))
except ValueError:
    pass
else:
    raise AssertionError("broadcasting incorrectly allowed")
assert regression_metrics(torch.tensor([2.]), torch.tensor([2.]))["rmse"] == 0

print("自检通过")


## Q504 完整回归循环与线性学习率边界

**演示改编，依据** [[课程/深度学习/演示代码/Chapter5.py]]、[[课程/深度学习/课堂笔记/2026-10-12-第11节-深度学习编程与分类模型宣讲.md#回归代码收尾与分类模型过渡]]。

用固定seed生成512/192条五维无噪声训练/测试点，x∈[-1,1]^5，y=1/(1+sum(x²))。
训练深度3、宽10 ReLU网络100步，Adam，学习率 `np.linspace(1e-2,1e-3,100)`。
接口 `linear_rates(start,end,steps)`：steps≥1，steps=1仅返回start；否则含两个端点。
每步清零→前向→MSE→backward→step。记录**更新前**loss为history504，结束后重算report504；画loss至output/Q504.png。
说明原Chapter5.py第三组range(N_iteration+1)为何访问5000项序列越界；自己的循环不得越界。

**自检接口／需保存的结果**

- `linear_rates`返回一维NumPy数组；将本次100步实际使用的学习率数组保存为`rates504`，形状`(100,)`。
- 已要求的`history504`为100个更新前训练MSE的Python float列表；`report504`为最终回归指标字典，键为`train_mse`、`train_rmse`、`test_mse`、`test_rmse`、`gap_abs`、`gap_signed`，值均为Python float。


<details><summary>提示（先尝试再展开）</summary>

linspace线性差值，不是乘固定倍率。枚举学习率本身可避免循环长度不同步。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def make_mlp(in_dim, out_dim=1, depth=3, width=100, activation="ReLU"):
    # depth counts Linear layers; hidden layers = depth - 1.
    if depth < 2 or width < 1:
        raise ValueError("depth >= 2 and width >= 1 required")
    activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
    if activation not in activations:
        raise ValueError("unknown activation")
    layers, last = [], in_dim
    for _ in range(depth - 1):
        layers += [nn.Linear(last, width), activations[activation]()]
        last = width
    layers += [nn.Linear(last, out_dim)]
    return nn.Sequential(*layers)

def regression_metrics(pred, target):
    # Deliberately reject broadcasting; squeeze(-1) is done at the caller.
    if pred.shape != target.shape or target.numel() == 0:
        raise ValueError("nonempty, equal shapes required")
    mse = ((pred - target) ** 2).mean().item()
    return {"mse": mse, "rmse": float(np.sqrt(mse))}

def regression_report(train_pred, train_y, test_pred, test_y):
    a = regression_metrics(train_pred, train_y)
    b = regression_metrics(test_pred, test_y)
    signed = b["rmse"] - a["rmse"]
    return {"train_mse": a["mse"], "train_rmse": a["rmse"],
            "test_mse": b["mse"], "test_rmse": b["rmse"],
            "gap_abs": abs(signed), "gap_signed": signed}

def linear_rates(start, end, steps):
    if steps < 1:
        raise ValueError("steps >= 1")
    return np.linspace(start, end, steps)

g504 = torch.Generator().manual_seed(504)
xt504 = 2 * torch.rand(512, 5, generator=g504) - 1
xv504 = 2 * torch.rand(192, 5, generator=g504) - 1
yt504 = 1 / (1 + xt504.square().sum(1))
yv504 = 1 / (1 + xv504.square().sum(1))
torch.manual_seed(504)
net504 = make_mlp(5, depth=3, width=10)
opt504 = torch.optim.Adam(net504.parameters(), lr=0.01)
rates504, history504 = linear_rates(0.01, 0.001, 100), []
for lr in rates504:  # length and iteration count cannot disagree
    for group in opt504.param_groups:
        group["lr"] = float(lr)
    opt504.zero_grad()
    loss = nn.functional.mse_loss(net504(xt504).squeeze(-1), yt504)
    loss.backward()
    opt504.step()
    history504.append(loss.item())
net504.eval()
with torch.no_grad():
    report504 = regression_report(net504(xt504).squeeze(-1), yt504,
                                  net504(xv504).squeeze(-1), yv504)
Path("output").mkdir(exist_ok=True)
fig, ax = plt.subplots()
ax.plot(history504)
ax.set(xlabel="Update", ylabel="Training MSE", title="Five-dimensional regression")
fig.savefig("output/Q504.png", dpi=120)
plt.close(fig)
print(report504)
print("Source bug: 5000 values -> legal indices 0..4999, not 5000")


### 解析与点评

原程序前两组range(5000)正确，第三组range(5001)多一次索引。损失曲线记录更新前的100个值，最终报告在第100次更新后另算；两者末项不必完全相同。未改教师源文件。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert len(history504) == 100 and np.isfinite(history504).all()
assert np.allclose(linear_rates(.1, .01, 3), [.1, .055, .01])
assert linear_rates(.1, .01, 1).tolist() == [.1]
assert np.isclose(rates504[0], .01) and np.isclose(rates504[-1], .001)
assert report504["train_mse"] < 1  # smoke check only, not a promised accuracy
assert Path("output/Q504.png").stat().st_size > 1000

print("自检通过")


## Q505 手动L1/L2：目标与评估损失分开

**新编，依据** [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]“正则化”小节、[[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#五、显式正则化]]。

实现 `penalty5(parameters,kind)`，kind为L1/L2，返回所有参数（**包括偏置**）的绝对值和/平方和；不得detach或.item破坏梯度。
令theta=[-2,0,3]、lambda=.2，分别求lambda*L1与lambda*L2的梯度，展示零点处PyTorch选用0次梯度。
再用1→8→1 ReLU网络对32个x∈[-1,1]的y=x²做60步SGD(.05)，objective=data_mse+.001*L2。
输出data_mse505、objective505，并说明optimizer weight_decay与手写惩罚不能不加说明重复使用。

**自检接口／需保存的结果**

- `grad_l1_505`、`grad_l2_505`：分别保存本题`lambda*L1`、`lambda*L2`对同一theta求导得到的梯度副本，形状均为`(3,)`的CPU float32 Tensor；两种梯度应单独保存，不能让后一轮梯度覆盖前一份结果。
- `net505`：完成60步训练后的1→8→1 ReLU网络，保留最后一次反传所得参数梯度以供检查。已要求的`data_mse505`、`objective505`均为最终模型重算得到的Python float。


<details><summary>提示（先尝试再展开）</summary>

L2附加梯度为2lambda*theta。总损失参与反传，报告RMSE时只用数据MSE开方。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def penalty5(parameters, kind="L2"):
    terms = list(parameters)
    if not terms or kind not in ("L1", "L2"):
        raise ValueError("nonempty parameters and L1/L2 required")
    if kind == "L1":
        return sum(p.abs().sum() for p in terms)
    return sum(p.square().sum() for p in terms)

theta505 = nn.Parameter(torch.tensor([-2., 0., 3.]))
(.2 * penalty5([theta505], "L1")).backward()
grad_l1_505 = theta505.grad.clone()
theta505.grad = None
(.2 * penalty5([theta505], "L2")).backward()
grad_l2_505 = theta505.grad.clone()
torch.manual_seed(505)
net505 = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))
x505 = torch.linspace(-1, 1, 32).reshape(-1, 1)
y505 = x505.squeeze(-1).square()
opt505 = torch.optim.SGD(net505.parameters(), lr=.05, weight_decay=0)
for _ in range(60):
    opt505.zero_grad()
    data_loss = (net505(x505).squeeze(-1)-y505).square().mean()
    objective = data_loss + .001*penalty5(net505.parameters(), "L2")
    objective.backward()
    opt505.step()
with torch.no_grad():
    data_mse505 = (net505(x505).squeeze(-1)-y505).square().mean().item()
    objective505 = data_mse505 + .001*penalty5(net505.parameters()).item()
print("L1 gradients:", grad_l1_505, "L2 gradients:", grad_l2_505)
print("Data MSE:", data_mse505, "Objective:", objective505)


### 解析与点评

L1“倾向稀疏”不表示普通有限步SGD必然得到精确零参数。这里按题面惩罚全体参数；实际工程也可只惩罚权重，但必须明确口径。与AdamW解耦衰减不能随意宣称等价。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert torch.allclose(grad_l1_505, torch.tensor([-.2, 0, .2]))
assert torch.allclose(grad_l2_505, torch.tensor([-.8, 0, 1.2]))
assert penalty5([torch.tensor([-2., 0., 3.])], "L1").item() == 5
assert penalty5([torch.tensor([-2., 0., 3.])], "L2").item() == 13
assert np.isfinite(objective505) and objective505 > data_mse505 >= 0
assert all(p.grad is not None for p in net505.parameters())

print("自检通过")


## Q506 早停：监控验证集并恢复最佳快照

**新编，依据** [[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#六、频谱偏置与早停]]。

补充解释：早停限制继续拟合，不是训练误差最小就停。实现 `EarlyStop5(patience=5,min_delta=0)`，`update(value,net)`在value比best小超过min_delta时clone每项state_dict；连续patience次未改善返回True。
实现 `restore(net)`，无快照时报ValueError。
用1维sin(2x)目标，固定seed生成96条有0.15高斯噪声训练数据、48条干净验证和48条独立干净测试数据。
1→16→1 Tanh网络，Adam(.02)，至多150步，验证MSE早停patience=12。恢复后仅做最终测试，输出best_epoch506、stop_epoch506、report506；允许预算内未触发早停。
另用人为验证序列[3,2,2.1,2.2]验证patience=2在第四项停止。

**自检接口／需保存的结果**

- `val_history506`：按实际更新顺序记录每步验证MSE的Python float列表；`restored_val506`：恢复最佳快照后重算的验证MSE，Python float。`best_epoch506`、`stop_epoch506`均为从1开始的Python整数。
- `report506`：恢复后计算的回归字典，键为`train_mse`、`train_rmse`、`test_mse`、`test_rmse`、`gap_abs`、`gap_signed`，值均为Python float。
- 人为序列检查另用`dummy506`（独立的1→1带偏置`nn.Linear`）和`probe506`（patience=2、min_delta=0的`EarlyStop5`实例）。`flags506`为依次输入题面四个验证值后返回的四个bool组成的列表。
- `EarlyStop5.state`应可访问，内容为最近最佳模型的`state_dict`快照字典（字符串键、Tensor值）；自检会修改`dummy506`当前权重，检查`probe506.state`不随之改变。


<details><summary>提示（先尝试再展开）</summary>

state_dict里的Tensor会随训练变化，字典浅复制不够；保存detach().clone()。验证数据不反传，测试不决定停止。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def make_mlp(in_dim, out_dim=1, depth=3, width=100, activation="ReLU"):
    # depth counts Linear layers; hidden layers = depth - 1.
    if depth < 2 or width < 1:
        raise ValueError("depth >= 2 and width >= 1 required")
    activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
    if activation not in activations:
        raise ValueError("unknown activation")
    layers, last = [], in_dim
    for _ in range(depth - 1):
        layers += [nn.Linear(last, width), activations[activation]()]
        last = width
    layers += [nn.Linear(last, out_dim)]
    return nn.Sequential(*layers)

def regression_metrics(pred, target):
    # Deliberately reject broadcasting; squeeze(-1) is done at the caller.
    if pred.shape != target.shape or target.numel() == 0:
        raise ValueError("nonempty, equal shapes required")
    mse = ((pred - target) ** 2).mean().item()
    return {"mse": mse, "rmse": float(np.sqrt(mse))}

def regression_report(train_pred, train_y, test_pred, test_y):
    a = regression_metrics(train_pred, train_y)
    b = regression_metrics(test_pred, test_y)
    signed = b["rmse"] - a["rmse"]
    return {"train_mse": a["mse"], "train_rmse": a["rmse"],
            "test_mse": b["mse"], "test_rmse": b["rmse"],
            "gap_abs": abs(signed), "gap_signed": signed}

class EarlyStop5:
    def __init__(self, patience=5, min_delta=0):
        if patience < 1 or min_delta < 0:
            raise ValueError("invalid early stopping settings")
        self.patience, self.min_delta = patience, min_delta
        self.best, self.wait, self.state = float("inf"), 0, None
    def update(self, value, net):
        if not np.isfinite(value):
            raise ValueError("validation loss must be finite")
        if value < self.best - self.min_delta:
            self.best, self.wait = value, 0
            self.state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            self.wait += 1
        return self.wait >= self.patience
    def restore(self, net):
        if self.state is None:
            raise ValueError("no checkpoint saved")
        net.load_state_dict(self.state)

torch.manual_seed(506)
x506 = 2*torch.rand(192, 1)-1
y506 = torch.sin(2*x506.squeeze(-1))
xt506, xv506, xs506 = x506[:96], x506[96:144], x506[144:]
yt506 = y506[:96] + .15*torch.randn(96)
yv506, ys506 = y506[96:144], y506[144:]
net506 = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))
optimizer506 = torch.optim.Adam(net506.parameters(), lr=.02)
stopper506, val_history506 = EarlyStop5(12), []
best_epoch506 = 0
for epoch in range(1, 151):
    net506.train()
    optimizer506.zero_grad()
    loss = (net506(xt506).squeeze(-1)-yt506).square().mean()
    loss.backward()
    optimizer506.step()
    net506.eval()
    with torch.no_grad():
        val = (net506(xv506).squeeze(-1)-yv506).square().mean().item()
    val_history506.append(val)
    improved = val < stopper506.best
    should_stop = stopper506.update(val, net506)
    if improved:
        best_epoch506 = epoch
    if should_stop:
        break
stop_epoch506 = epoch
stopper506.restore(net506)
with torch.no_grad():
    restored_val506 = (net506(xv506).squeeze(-1)-yv506).square().mean().item()
    report506 = regression_report(net506(xt506).squeeze(-1), yt506,
                                 net506(xs506).squeeze(-1), ys506)
dummy506 = nn.Linear(1, 1)
probe506 = EarlyStop5(patience=2)
flags506 = [probe506.update(v, dummy506) for v in [3, 2, 2.1, 2.2]]
print("Best/stop epoch:", best_epoch506, stop_epoch506, "Final report:", report506)


### 解析与点评

最佳轮数与停止轮数不同。题目检查人为loss序列和快照独立性，避免仅凭某次随机训练恰好提前停止。no_grad关闭求导，eval切换模块运行模式，二者职责不同。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert flags506 == [False, False, False, True]
assert 1 <= best_epoch506 <= stop_epoch506 <= 150
assert np.isclose(restored_val506, min(val_history506), rtol=1e-5)
assert np.isfinite(list(report506.values())).all()
old506 = {k: v.clone() for k, v in probe506.state.items()}
with torch.no_grad():
    dummy506.weight.add_(100)
assert all(torch.equal(old506[k], probe506.state[k]) for k in old506)

print("自检通过")


## Q507 随机初始化与神经元对称性

**新编，依据** [[课程/深度学习/PPT/Chapter5 (1)/Chapter5 (1).md]]“变量(网络参数)初始化”小节、[[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#三、参数初始化与对称性]]。

对2→4→1 ReLU网络，在同层所有weight=.2、bias=.1的情况下做一步SGD(.1)，检查第一隐藏层四行权重保持相同。
另建随机权重但bias全0的同结构网络，验证各神经元权重并不相同。说明“权重需打破对称性”不意味着“每个偏置都必须随机”。
补充API：为ReLU隐藏层调用`nn.init.kaiming_normal_`（fan_in），输出层调用`nn.init.xavier_uniform_`，仅检查有限、形状不变，不强求小样本方差等于理论值。
命名same507、random507、max_rowdiff507。

**自检接口补充**

已要求的`same507`与`random507`均保存为三项`nn.Sequential`（索引0为2→4 Linear，索引1为ReLU，索引2为4→1 Linear）。`max_rowdiff507`是完成一次更新后，`same507`第一层权重各行相对第一行的最大绝对差，保存为Python float；不是梯度差或两网络间差值。


<details><summary>提示（先尝试再展开）</summary>

同层四个神经元前向与反向完全对称；随机权重打破对称性，零偏置仍可用。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

torch.manual_seed(507)
same507 = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
with torch.no_grad():
    for layer in (same507[0], same507[2]):
        layer.weight.fill_(.2)
        layer.bias.fill_(.1)
x507 = torch.tensor([[1., 2.], [2., 1.], [-1., 2.]])
y507 = torch.tensor([1., 0., .5])
opt507 = torch.optim.SGD(same507.parameters(), lr=.1)
opt507.zero_grad()
(same507(x507).squeeze(-1)-y507).square().mean().backward()
opt507.step()
max_rowdiff507 = (same507[0].weight-same507[0].weight[0]).abs().max().item()
random507 = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
with torch.no_grad():
    nn.init.kaiming_normal_(random507[0].weight, mode="fan_in", nonlinearity="relu")
    nn.init.xavier_uniform_(random507[2].weight)
    random507[0].bias.zero_()
    random507[2].bias.zero_()
print("Identical-row difference after update:", max_rowdiff507)
print("Random hidden weights:\n", random507[0].weight)


### 解析与点评

本题是特定同值初始化的可运行例证，不宣称所有常数初始化都造成相同故障。课件介绍LeCun/Xavier/He不表示原Chapter5.py已经调用这些初始化API；原代码使用Linear默认初始化。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert max_rowdiff507 < 1e-7
assert not torch.allclose(random507[0].weight[0], random507[0].weight[1])
assert torch.equal(random507[0].bias, torch.zeros(4))
assert all(torch.isfinite(p).all() for p in random507.parameters())
assert random507(torch.zeros(1, 2)).shape == (1, 1)

print("自检通过")


## Q508 小批量与末批权重：Adam不是自动切批

**新编，依据** [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#A-10 Adam就等于小批量吗]]、[[课程/深度学习/课堂笔记/2026-09-23-第8节-深度学习模型训练与正则化讲座.md#七、PyTorch 网络实现]]。

对130个1维样本，batch_size=32，训练1→8→1 ReLU网络两轮，SGD(.03)，每轮用randperm打乱。
记录batch_sizes508（每轮5步、末批2条）与updates508（共10次更新）。
实现 `batched_mse5(net,x,y,batch_size)`，在no_grad下按每批**平方误差和/总样本数**汇总；结果要等于全量MSE，不能对5个批均值直接平均。
说明原Chapter5.py每步用全训练集，并非mini-batch；优化器名称不决定批量。

**自检接口／需保存的结果**

`batched508`：训练完成后，用`batched_mse5`对全部130条训练数据得到的MSE；`full508`：同一最终网络、同一批数据一次性评估得到的MSE，二者均保存为Python float。已要求的`batch_sizes508`为按轮排列的二维Python整数列表，`updates508`为累计参数更新次数的Python整数。


<details><summary>提示（先尝试再展开）</summary>

末批大小不等于32，按样本数加权，或者直接累积平方和。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def batched_mse5(net, x, y, batch_size=32):
    if len(y) == 0 or batch_size < 1:
        raise ValueError("nonempty data, positive batch size")
    total = 0.0
    with torch.no_grad():
        for start in range(0, len(y), batch_size):
            stop = start + batch_size
            total += (net(x[start:stop]).squeeze(-1)-y[start:stop]).square().sum().item()
    return total / len(y)

torch.manual_seed(508)
x508 = torch.linspace(-1, 1, 130).reshape(-1, 1)
y508 = x508.squeeze(-1).square()
net508 = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))
opt508 = torch.optim.SGD(net508.parameters(), lr=.03)
batch_sizes508, updates508 = [], 0
for _ in range(2):
    permutation = torch.randperm(len(y508))
    sizes = []
    for start in range(0, len(y508), 32):
        idx = permutation[start:start+32]
        opt508.zero_grad()
        loss = (net508(x508[idx]).squeeze(-1)-y508[idx]).square().mean()
        loss.backward()
        opt508.step()
        sizes.append(len(idx))
        updates508 += 1
    batch_sizes508.append(sizes)
net508.eval()
batched508 = batched_mse5(net508, x508, y508)
with torch.no_grad():
    full508 = (net508(x508).squeeze(-1)-y508).square().mean().item()
print("Batch sizes:", batch_sizes508, "updates:", updates508, "MSE:", batched508)


### 解析与点评

epoch是扫过全部训练样本一次，step是参数更新一次；本例2轮=10步。小批量训练每批均值可作为各步目标，但最终全数据评估的批均值必须按样本数加权。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert batch_sizes508 == [[32, 32, 32, 32, 2]]*2 and updates508 == 10
assert np.isclose(batched508, full508, rtol=1e-5, atol=1e-7)
class Identity508(nn.Module):
    def forward(self, x):
        return x
# Diagnostic unequal batches: two zero residuals and one residual 3 -> MSE 3, not 4.5.
assert np.isclose(batched_mse5(Identity508(), torch.tensor([[0.],[0.],[3.]]), torch.zeros(3), 2), 3)

print("自检通过")


## Q509 Homework03 Exercise 1：独立可跑的10维回归

**作业改编，精确来源** [[课程/深度学习/作业/Homework03/Homework03.md]] Exercise 1 (1)—(2)，[[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md#C5-07 高维回归的完整演示]]。

**资料缺口**：未找到原作业两份`.data`，不能复现未知原目标。本题默认**新编合成10维替代数据**，不冒充原作业数据；原训练规模100万点，本地默认256/96点，允许按设备选子集。

实现 `load_regression_data(path=None,*,trusted_pickle=False,n_train=256,n_test=96,seed=509)`。
path无/不存在时，x均匀于[-1,1]^10，替代y=1/(1+sum(x²))+.05*sin(3x0)，返回x_train、y_train、x_test、y_test及origin并打印来源。
若显式指定现有pickle，必须先拒绝未授权的trusted_pickle=False；只有用户核实本地来源后设True才可读取。预期键如上，校验X=(N,10)、y=(N,)、有限且非空。**不要下载或载入陌生pickle**。

实现 `make_mlp`与`fit_regression`。基准深度3（两隐藏层）、宽100、ReLU、输出无激活、MSE、SGD(lr=.03)，100步全批。
输出训练损失MSE、训练/测试RMSE、课程gap_abs，以及诊断gap_signed；返回(net,history,report)。数据、初始参数固定seed，CPU单线程。不保证达到原作业性能。

**自检接口／需保存的结果**

- 自检先跑默认离线合成路径，将加载结果保存为`data509`字典：`x_train`为float32 Tensor `(256,10)`，`y_train`为`(256,)`；`x_test`为`(96,10)`，`y_test`为`(96,)`；`origin`为打印的数据来源字符串，须包含大写`SYNTHETIC`以明确替代来源。
- 将基准拟合返回的网络、100个更新前MSE组成的Python float列表、报告字典分别保存为`net509`、`history509`、`report509`。网络批量输出`(N,1)`。
- `report509`的键为`train_mse`、`train_rmse`、`test_mse`、`test_rmse`、`gap_abs`、`gap_signed`（Python float）及`parameter_count`（Python int）。读取可信原文件的尝试与默认合成路径分开，避免改变本题默认自检规模。


<details><summary>提示（先尝试再展开）</summary>

深度=Linear数；SGD全批就是最速下降的数值实现。pickle读取前先检查显式信任参数。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def make_mlp(in_dim, out_dim=1, depth=3, width=100, activation="ReLU"):
    # depth counts Linear layers; hidden layers = depth - 1.
    if depth < 2 or width < 1:
        raise ValueError("depth >= 2 and width >= 1 required")
    activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
    if activation not in activations:
        raise ValueError("unknown activation")
    layers, last = [], in_dim
    for _ in range(depth - 1):
        layers += [nn.Linear(last, width), activations[activation]()]
        last = width
    layers += [nn.Linear(last, out_dim)]
    return nn.Sequential(*layers)

def regression_metrics(pred, target):
    # Deliberately reject broadcasting; squeeze(-1) is done at the caller.
    if pred.shape != target.shape or target.numel() == 0:
        raise ValueError("nonempty, equal shapes required")
    mse = ((pred - target) ** 2).mean().item()
    return {"mse": mse, "rmse": float(np.sqrt(mse))}

def regression_report(train_pred, train_y, test_pred, test_y):
    a = regression_metrics(train_pred, train_y)
    b = regression_metrics(test_pred, test_y)
    signed = b["rmse"] - a["rmse"]
    return {"train_mse": a["mse"], "train_rmse": a["rmse"],
            "test_mse": b["mse"], "test_rmse": b["rmse"],
            "gap_abs": abs(signed), "gap_signed": signed}

def load_regression_data(path=None, *, trusted_pickle=False, n_train=256, n_test=96, seed=509):
    """A local opt-in adapter, not an automatic untrusted pickle reader."""
    if path is not None and Path(path).is_file():
        if not trusted_pickle:
            raise ValueError("pickle can execute code: inspect provenance before explicit opt-in")
        import pickle
        with Path(path).open("rb") as f:
            raw = pickle.load(f)  # ONLY the caller's explicitly trusted local file.
        origin = "TRUSTED local pickle: " + str(Path(path).resolve())
    else:
        g = torch.Generator().manual_seed(seed)
        def sample(n):
            x = 2 * torch.rand(n, 10, generator=g) - 1
            # Newly defined replacement target; not the unknown homework function.
            y = 1 / (1 + x.square().sum(1)) + 0.05 * torch.sin(3 * x[:, 0])
            return x, y
        xt, yt = sample(n_train)
        xv, yv = sample(n_test)
        raw = dict(x_train=xt, y_train=yt, x_test=xv, y_test=yv)
        origin = "SYNTHETIC 10D regression replacement; original .data unavailable/not supplied"
    data = {}
    for split in ("train", "test"):
        x = torch.as_tensor(raw["x_" + split], dtype=torch.float32).clone()
        y = torch.as_tensor(raw["y_" + split], dtype=torch.float32).reshape(-1).clone()
        if x.ndim != 2 or x.shape[1] != 10 or len(x) != len(y) or len(y) == 0:
            raise ValueError("expected X=(N,10), y=(N,), N>0")
        if not torch.isfinite(x).all() or not torch.isfinite(y).all():
            raise ValueError("nonfinite data")
        data["x_" + split], data["y_" + split] = x, y
    data["origin"] = origin
    print("Data source:", origin)
    return data

def fit_regression(data, *, depth=3, width=100, activation="ReLU", optimizer_name="SGD",
                   lr=0.03, steps=100, seed=509):
    if steps < 1:
        raise ValueError("steps must be positive")
    torch.manual_seed(seed)
    net = make_mlp(10, 1, depth, width, activation)
    optimizers = {"SGD": torch.optim.SGD, "Adagrad": torch.optim.Adagrad,
                  "RMSprop": torch.optim.RMSprop, "Adam": torch.optim.Adam}
    optimizer = optimizers[optimizer_name](net.parameters(), lr=lr)
    loss_fn, history = nn.MSELoss(), []
    x, y = data["x_train"], data["y_train"]
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(net(x).squeeze(-1), y)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        report = regression_report(net(x).squeeze(-1), y,
                                   net(data["x_test"]).squeeze(-1), data["y_test"])
    report["parameter_count"] = sum(p.numel() for p in net.parameters())
    return net, history, report

data509 = load_regression_data()
net509, history509, report509 = fit_regression(data509)
print("Homework Exercise 1 replacement baseline:", report509)


### 解析与点评

基准参数量10×100+100 +100×100+100 +100×1+1=11301。来源标记必须随结果保留。真实文件可能采用不同键或存储格式，需先核对而不能盲猜；这里适配器只支持题面四键字典。测试集不进入loss反传。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert data509["x_train"].shape == (256, 10) and data509["y_test"].shape == (96,)
assert "SYNTHETIC" in data509["origin"]
assert sum(isinstance(m, nn.Linear) for m in net509.modules()) == 3
assert report509["parameter_count"] == 11301
assert len(history509) == 100 and np.isfinite(history509).all()
assert np.isclose(report509["gap_abs"], abs(report509["test_rmse"]-report509["train_rmse"]))
assert net509(torch.zeros(1, 10)).shape == (1, 1)
# Existing untrusted local file is refused before pickle parsing, even if invalid bytes.
import tempfile
with tempfile.TemporaryDirectory() as temp509:
    bad = Path(temp509)/"untrusted.data"
    bad.write_bytes(b"not a pickle")
    try:
        load_regression_data(bad)
    except ValueError as exc:
        assert "pickle" in str(exc)
    else:
        raise AssertionError("untrusted input was not rejected")

print("自检通过")


## Q510 Homework03 Exercise 1：完整单因素对照入口

**作业改编，精确来源** [[课程/深度学习/作业/Homework03/Homework03.md]] Exercise 1 (3)—(6)。本题为学习期扩展，**不要求在30分钟模拟卷中做完**。

实现 `comparison_configs()` 与 `run_reg_comparisons(data=None,steps=30,lr=.03)`。
基准(深度3,宽100,ReLU,SGD)。逐一只改一个因素：深度2/3/4/5；宽50/100/150/200；激活sigmoid/softplus/ReLU；优化器SGD/Adagrad/RMSprop/Adam，**学习率相同**。
共享基准只运行一次，因此共12种唯一配置（基准1+深度3+宽度3+激活2+优化器3）。返回每行含factor、value、全部配置、steps、lr、训练MSE、train/test RMSE、gap_abs、参数量，保存output/Q510-comparison.json并画测试RMSE柱图。
默认合成256/96、每配置30步、CPU；可显式加大steps、传可信原数据子集。重置同seed、共享数据和预算；同结构优化器对照应具有相同初始权重。记录现象，不宣称某算法必胜；用验证集选超参，若反复用测试表选最优则需另留最终测试集。

**自检接口／需保存的结果**

将默认`steps=30,lr=.03`完整比较的返回值保存为`rows510`：长度12的Python字典列表，**第一行是共享基准**，其余每行只改变一个配置因素。每行约定以下键与内容：

- `factor`、`value`：变化因素名称及所取选项；`origin`：数据来源字符串。
- `depth`、`width`：整数；`activation`：`ReLU`、`sigmoid`、`softplus`之一；`optimizer_name`：`SGD`、`Adagrad`、`RMSprop`、`Adam`之一。
- `steps`为更新数（int），`lr`为学习率（float）；`parameter_count`为参数总数（int）。
- `train_mse`、`train_rmse`、`test_mse`、`test_rmse`、`gap_abs`、`gap_signed`为该配置最终评估的Python float，不把训练历史列表放进这些指标字段。


<details><summary>提示（先尝试再展开）</summary>

不要做4×4×3×4笛卡尔积；单因素是在同一基准上每次只改一项。

</details>


In [ ]:
import numpy as np
import torch
from torch import nn
from pathlib import Path
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(2026)

def make_mlp(in_dim, out_dim=1, depth=3, width=100, activation="ReLU"):
    # depth counts Linear layers; hidden layers = depth - 1.
    if depth < 2 or width < 1:
        raise ValueError("depth >= 2 and width >= 1 required")
    activations = {"ReLU": nn.ReLU, "sigmoid": nn.Sigmoid, "softplus": nn.Softplus}
    if activation not in activations:
        raise ValueError("unknown activation")
    layers, last = [], in_dim
    for _ in range(depth - 1):
        layers += [nn.Linear(last, width), activations[activation]()]
        last = width
    layers += [nn.Linear(last, out_dim)]
    return nn.Sequential(*layers)

def regression_metrics(pred, target):
    # Deliberately reject broadcasting; squeeze(-1) is done at the caller.
    if pred.shape != target.shape or target.numel() == 0:
        raise ValueError("nonempty, equal shapes required")
    mse = ((pred - target) ** 2).mean().item()
    return {"mse": mse, "rmse": float(np.sqrt(mse))}

def regression_report(train_pred, train_y, test_pred, test_y):
    a = regression_metrics(train_pred, train_y)
    b = regression_metrics(test_pred, test_y)
    signed = b["rmse"] - a["rmse"]
    return {"train_mse": a["mse"], "train_rmse": a["rmse"],
            "test_mse": b["mse"], "test_rmse": b["rmse"],
            "gap_abs": abs(signed), "gap_signed": signed}

def load_regression_data(path=None, *, trusted_pickle=False, n_train=256, n_test=96, seed=509):
    """A local opt-in adapter, not an automatic untrusted pickle reader."""
    if path is not None and Path(path).is_file():
        if not trusted_pickle:
            raise ValueError("pickle can execute code: inspect provenance before explicit opt-in")
        import pickle
        with Path(path).open("rb") as f:
            raw = pickle.load(f)  # ONLY the caller's explicitly trusted local file.
        origin = "TRUSTED local pickle: " + str(Path(path).resolve())
    else:
        g = torch.Generator().manual_seed(seed)
        def sample(n):
            x = 2 * torch.rand(n, 10, generator=g) - 1
            # Newly defined replacement target; not the unknown homework function.
            y = 1 / (1 + x.square().sum(1)) + 0.05 * torch.sin(3 * x[:, 0])
            return x, y
        xt, yt = sample(n_train)
        xv, yv = sample(n_test)
        raw = dict(x_train=xt, y_train=yt, x_test=xv, y_test=yv)
        origin = "SYNTHETIC 10D regression replacement; original .data unavailable/not supplied"
    data = {}
    for split in ("train", "test"):
        x = torch.as_tensor(raw["x_" + split], dtype=torch.float32).clone()
        y = torch.as_tensor(raw["y_" + split], dtype=torch.float32).reshape(-1).clone()
        if x.ndim != 2 or x.shape[1] != 10 or len(x) != len(y) or len(y) == 0:
            raise ValueError("expected X=(N,10), y=(N,), N>0")
        if not torch.isfinite(x).all() or not torch.isfinite(y).all():
            raise ValueError("nonfinite data")
        data["x_" + split], data["y_" + split] = x, y
    data["origin"] = origin
    print("Data source:", origin)
    return data

def fit_regression(data, *, depth=3, width=100, activation="ReLU", optimizer_name="SGD",
                   lr=0.03, steps=100, seed=509):
    if steps < 1:
        raise ValueError("steps must be positive")
    torch.manual_seed(seed)
    net = make_mlp(10, 1, depth, width, activation)
    optimizers = {"SGD": torch.optim.SGD, "Adagrad": torch.optim.Adagrad,
                  "RMSprop": torch.optim.RMSprop, "Adam": torch.optim.Adam}
    optimizer = optimizers[optimizer_name](net.parameters(), lr=lr)
    loss_fn, history = nn.MSELoss(), []
    x, y = data["x_train"], data["y_train"]
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(net(x).squeeze(-1), y)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    net.eval()
    with torch.no_grad():
        report = regression_report(net(x).squeeze(-1), y,
                                   net(data["x_test"]).squeeze(-1), data["y_test"])
    report["parameter_count"] = sum(p.numel() for p in net.parameters())
    return net, history, report

def comparison_configs():
    base = dict(depth=3, width=100, activation="ReLU", optimizer_name="SGD")
    rows = [("baseline", "default", base.copy())]
    factors = {"depth": (2, 4, 5), "width": (50, 150, 200),
               "activation": ("sigmoid", "softplus"),
               "optimizer_name": ("Adagrad", "RMSprop", "Adam")}
    for factor, values in factors.items():
        for value in values:
            cfg = base.copy()
            cfg[factor] = value
            rows.append((factor, value, cfg))
    return rows

import json
def run_reg_comparisons(data=None, steps=30, lr=.03):
    if data is None:
        data = load_regression_data()
    result = []
    for factor, value, config in comparison_configs():
        _, _, report = fit_regression(data, **config, steps=steps, lr=lr, seed=510)
        result.append(dict(factor=factor, value=value, origin=data["origin"], **config, lr=lr, steps=steps, **report))
    return result

rows510 = run_reg_comparisons()
Path("output").mkdir(exist_ok=True)
Path("output/Q510-comparison.json").write_text(json.dumps(rows510, indent=2), encoding="utf-8")
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(len(rows510)), [r["test_rmse"] for r in rows510])
ax.set_xticks(range(len(rows510)), [r["factor"]+":"+str(r["value"]) for r in rows510], rotation=65, ha="right")
ax.set(ylabel="Test RMSE", title="Single-factor comparison (synthetic replacement)")
fig.tight_layout()
fig.savefig("output/Q510.png", dpi=120)
plt.close(fig)
for row in rows510:
    print(row["factor"], row["value"], "test RMSE", round(row["test_rmse"], 5))


### 解析与点评

12种配置已覆盖所有指定选项，未把共享基准重复算四次。30步只核对实验流程，尤其深层sigmoid可能训练不足；不能据轻量合成数据表推断原100万点任务结论。相同学习率是作业控制条件，不等于各优化器的最佳学习率相同。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert len(rows510) == 12
assert {r["depth"] for r in rows510} == {2,3,4,5}
assert {r["width"] for r in rows510} == {50,100,150,200}
assert {r["activation"] for r in rows510} == {"ReLU","sigmoid","softplus"}
assert {r["optimizer_name"] for r in rows510} == {"SGD","Adagrad","RMSprop","Adam"}
base510 = rows510[0]
keys510 = ["depth", "width", "activation", "optimizer_name"]
assert all(sum(r[k] != base510[k] for k in keys510) == 1 for r in rows510[1:])
assert all(r["lr"] == .03 and r["steps"] == 30 for r in rows510)
assert all(np.isfinite(r["test_rmse"]) for r in rows510)
assert Path("output/Q510-comparison.json").is_file()

print("自检通过")
